In [1]:
# ============================================================
# D12 — Branch C: Normalised Markdown conversion
# 0. Imports and frozen experimental configuration
# ============================================================

import json
import hashlib
import platform
import re
import sys
import unicodedata

from collections import Counter
from datetime import datetime
from pathlib import Path

from google.colab import files

DOCUMENT_ID = "D12"
DOCUMENT_NAME = "Our World in Data — Annual CO2 emissions time series"

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

SOURCE_FORMAT = ".csv"

EXPECTED_SOURCE_SHA256 = (
    "eb46e8e036c08032bbd18d234d3da22442111288e26af32d3eacef592457e788"
)

EXPECTED_BRANCH_B_REPRESENTATION_SHA256 = (
    "720444ec3a499891ada2dcd5dc046aec6cd25cadf9aac68d689834be0158f941"
)

EXPECTED_SOURCE_ROW_COUNT = 275
EXPECTED_MIN_YEAR = 1750
EXPECTED_MAX_YEAR = 2024

TARGET_YEARS = [
    1750, 1800, 1850, 1900, 1950,
    1960, 1970, 1980, 1990, 2000,
    2010, 2011, 2012, 2013, 2014,
    2015, 2016, 2017, 2018, 2019,
    2020, 2021, 2022, 2023, 2024
]

EXPECTED_RECORD_COUNT = len(TARGET_YEARS)

REFERENCE_CATEGORY = "Environmental time-series"
REFERENCE_TOPIC = "Annual CO2 emissions"
REFERENCE_DESCRIPTION = "Annual CO2 emissions"
REFERENCE_UNIT = None

EXPECTED_CATEGORY_COUNTS = {
    REFERENCE_CATEGORY:
        EXPECTED_RECORD_COUNT
}

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]

STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Reporting Period",
    "Source Location"
]

VALUE_ALLOWED_TYPES = (
    str,
    int,
    float,
    type(None)
)

OUTPUT_DIR = Path("outputs_D12_branch_C")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARENT_CHECK_PATH = (
    OUTPUT_DIR / "D12_branch_C_parent_B_equivalence_check.json"
)

NORMALISATION_CHECK_PATH = (
    OUTPUT_DIR / "D12_branch_C_normalisation_check.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR / "D12_branch_C_normalised_markdown.md"
)

REPRESENTATION_METADATA_PATH = (
    OUTPUT_DIR / "D12_branch_C_representation_metadata.json"
)

PROMPT_PATH = (
    OUTPUT_DIR / "D12_branch_C_prompt.txt"
)

EXPERIMENT_METADATA_PRE_PATH = (
    OUTPUT_DIR / "D12_branch_C_experiment_metadata_pre.json"
)

PRECHECK_PATH = (
    OUTPUT_DIR / "D12_branch_C_pre_extraction_check.json"
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR / "D12_branch_C_raw_response.txt"
)

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR / "D12_branch_C_parsed_extraction.json"
)

STRUCTURE_CHECK_PATH = (
    OUTPUT_DIR / "D12_branch_C_structure_check.json"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR / "D12_branch_C_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR / "D12_branch_C_experiment_summary.json"
)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Parent branch:", PARENT_BRANCH)
print("Expected source rows:", EXPECTED_SOURCE_ROW_COUNT)


Document: D12
Branch: C
Parent branch: B
Expected source rows: 275


In [2]:
# ============================================================
# 1. Upload original D12 CSV and required frozen Branch B artefacts
# ============================================================
#
# Upload exactly:
#   1) original D12 CSV
#   2) D12_branch_B_structural_markdown.md
#   3) D12_branch_B_conversion_integrity.json
#
# Branch C deliberately does NOT recreate Branch B.
# ============================================================

uploaded = files.upload()
names = list(uploaded.keys())

csv_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".csv")
]

md_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".md")
]

json_files = [
    Path(name)
    for name in names
    if name.lower().endswith(".json")
]

if (
    len(csv_files) != 1
    or len(md_files) != 1
    or len(json_files) != 1
):
    raise ValueError(
        "Upload exactly one D12 CSV, one frozen Branch B structural "
        "Markdown file, and one Branch B conversion-integrity JSON file."
    )

SOURCE_PATH = csv_files[0]
BRANCH_B_REPRESENTATION_PATH = md_files[0]
BRANCH_B_CHECK_PATH = json_files[0]

print("Source:", SOURCE_PATH.name)
print("Branch B representation:", BRANCH_B_REPRESENTATION_PATH.name)
print("Branch B integrity:", BRANCH_B_CHECK_PATH.name)


Saving D12_branch_B_conversion_integrity.json to D12_branch_B_conversion_integrity.json
Saving D12_branch_B_structural_markdown.md to D12_branch_B_structural_markdown.md
Saving D12 - ourworldindataCO2.csv to D12 - ourworldindataCO2.csv
Source: D12 - ourworldindataCO2.csv
Branch B representation: D12_branch_B_structural_markdown.md
Branch B integrity: D12_branch_B_conversion_integrity.json


In [3]:
# ============================================================
# 2. Verify frozen source identity and Branch B provenance
# ============================================================

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(chunk_size),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def sha256_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)

SOURCE_HASH_MATCH = (
    SOURCE_SHA256
    == EXPECTED_SOURCE_SHA256
)

if not SOURCE_HASH_MATCH:
    raise ValueError(
        "Uploaded D12 CSV does not match the frozen Stage 1 source identity."
    )


with open(
    BRANCH_B_CHECK_PATH,
    "r",
    encoding="utf-8"
) as f:
    branch_b_check = json.load(f)


if branch_b_check.get("document_id") != DOCUMENT_ID:
    raise ValueError(
        "Branch B conversion-integrity artefact belongs to another document."
    )

if branch_b_check.get("branch") != "B":
    raise ValueError(
        "Uploaded conversion-integrity artefact is not from Branch B."
    )

if branch_b_check.get("source_sha256") != SOURCE_SHA256:
    raise ValueError(
        "Branch B conversion-integrity artefact refers to another D12 source."
    )

if not branch_b_check.get(
    "conversion_integrity_passed",
    False
):
    raise ValueError(
        "The frozen D12 Branch B representation did not pass conversion integrity."
    )


SOURCE_B_MARKDOWN = (
    BRANCH_B_REPRESENTATION_PATH.read_text(
        encoding="utf-8"
    )
)

if not SOURCE_B_MARKDOWN.strip():
    raise ValueError(
        "Uploaded Branch B structural Markdown is empty."
    )


UPLOADED_BRANCH_B_SHA256 = sha256_text(
    SOURCE_B_MARKDOWN
)

BRANCH_B_HASH_MATCH = (
    UPLOADED_BRANCH_B_SHA256
    == EXPECTED_BRANCH_B_REPRESENTATION_SHA256
)

if not BRANCH_B_HASH_MATCH:
    raise ValueError(
        "Uploaded Branch B Markdown does not match the frozen final "
        "D12 Branch B representation SHA-256."
    )


print("Frozen source identity verified.")
print("Branch B conversion provenance verified.")
print("Frozen Branch B SHA-256 verified:", BRANCH_B_HASH_MATCH)


Frozen source identity verified.
Branch B conversion provenance verified.
Frozen Branch B SHA-256 verified: True


In [4]:
# ============================================================
# 3. Verify exact Branch B parent equivalence
# ============================================================

TABLE_ROW_PATTERN = re.compile(
    r"^\|\s*(\d+)\s*\|\s*(\d{4})\s*\|\s*(-?\d+)\s*\|$"
)

parent_rows = []

for line in SOURCE_B_MARKDOWN.splitlines():

    match = TABLE_ROW_PATTERN.fullmatch(
        line.strip()
    )

    if match:
        parent_rows.append({
            "CSV data row":
                int(match.group(1)),

            "Year":
                int(match.group(2)),

            "Annual CO2 emissions":
                int(match.group(3))
        })


parent_row_count_valid = (
    len(parent_rows)
    == EXPECTED_SOURCE_ROW_COUNT
)

parent_years = [
    row["Year"]
    for row in parent_rows
]

parent_values = [
    row["Annual CO2 emissions"]
    for row in parent_rows
]

parent_source_rows = [
    row["CSV data row"]
    for row in parent_rows
]

parent_year_sequence_valid = (
    parent_years
    == list(
        range(
            EXPECTED_MIN_YEAR,
            EXPECTED_MAX_YEAR + 1
        )
    )
)

parent_source_row_sequence_valid = (
    parent_source_rows
    == list(
        range(
            2,
            EXPECTED_SOURCE_ROW_COUNT + 2
        )
    )
)

parent_years_unique = (
    len(parent_years)
    == len(set(parent_years))
)

all_target_years_present = all(
    year in set(parent_years)
    for year in TARGET_YEARS
)


PARENT_EQUIVALENCE_PASSED = bool(
    SOURCE_HASH_MATCH
    and branch_b_check.get(
        "conversion_integrity_passed",
        False
    )
    and BRANCH_B_HASH_MATCH
    and parent_row_count_valid
    and parent_year_sequence_valid
    and parent_source_row_sequence_valid
    and parent_years_unique
    and all_target_years_present
)


parent_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_sha256":
        SOURCE_SHA256,

    "source_hash_matches_frozen_identity":
        SOURCE_HASH_MATCH,

    "branch_B_conversion_integrity_passed":
        bool(
            branch_b_check.get(
                "conversion_integrity_passed",
                False
            )
        ),

    "expected_frozen_branch_B_sha256":
        EXPECTED_BRANCH_B_REPRESENTATION_SHA256,

    "uploaded_branch_B_sha256":
        UPLOADED_BRANCH_B_SHA256,

    "uploaded_branch_B_hash_matches_frozen_parent":
        BRANCH_B_HASH_MATCH,

    "parent_row_count":
        len(parent_rows),

    "parent_row_count_valid":
        parent_row_count_valid,

    "parent_year_sequence_valid":
        parent_year_sequence_valid,

    "parent_source_row_sequence_valid":
        parent_source_row_sequence_valid,

    "parent_years_unique":
        parent_years_unique,

    "all_target_years_present":
        all_target_years_present,

    "parent_equivalence_method":
        (
            "Frozen Branch B representation SHA-256 + Branch B "
            "conversion-integrity provenance; Branch B is not regenerated"
        ),

    "branch_B_regeneration_attempted":
        False,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED
}


PARENT_CHECK_PATH.write_text(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        parent_check,
        ensure_ascii=False,
        indent=2
    )
)


if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError(
        "D12 Branch C parent-equivalence verification failed."
    )


{
  "document_id": "D12",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "eb46e8e036c08032bbd18d234d3da22442111288e26af32d3eacef592457e788",
  "source_hash_matches_frozen_identity": true,
  "branch_B_conversion_integrity_passed": true,
  "expected_frozen_branch_B_sha256": "720444ec3a499891ada2dcd5dc046aec6cd25cadf9aac68d689834be0158f941",
  "uploaded_branch_B_sha256": "720444ec3a499891ada2dcd5dc046aec6cd25cadf9aac68d689834be0158f941",
  "uploaded_branch_B_hash_matches_frozen_parent": true,
  "parent_row_count": 275,
  "parent_row_count_valid": true,
  "parent_year_sequence_valid": true,
  "parent_source_row_sequence_valid": true,
  "parent_years_unique": true,
  "all_target_years_present": true,
  "parent_equivalence_method": "Frozen Branch B representation SHA-256 + Branch B conversion-integrity provenance; Branch B is not regenerated",
  "branch_B_regeneration_attempted": false,
  "parent_equivalence_passed": true
}


In [5]:
# ============================================================
# 4. Define conservative deterministic Branch C normalisation
# ============================================================
#
# D12 Branch B is already regular. Branch C therefore uses only
# non-semantic textual normalisation.
#
# Allowed:
# - Unicode NFKC;
# - Unicode-space standardisation;
# - typographic apostrophe standardisation;
# - dash/minus-glyph standardisation;
# - soft-hyphen removal;
# - line-ending standardisation;
# - horizontal whitespace normalisation;
# - excessive blank-line standardisation.
#
# NOT applied:
# - source-row filtering;
# - target-year filtering;
# - year/value rewriting;
# - numeric conversion or rescaling;
# - unit inference;
# - semantic relabelling;
# - manual correction;
# - reference-guided repair.
# ============================================================

UNICODE_SPACE_CHARACTERS = [
    "\u00a0", "\u1680", "\u2000", "\u2001", "\u2002",
    "\u2003", "\u2004", "\u2005", "\u2006", "\u2007",
    "\u2008", "\u2009", "\u200a", "\u202f", "\u205f",
    "\u3000"
]

APOSTROPHE_REPLACEMENTS = {
    "’": "'",
    "‘": "'",
    "‛": "'",
    "´": "'",
    "`": "'"
}

DASH_REPLACEMENTS = {
    "‐": "-",
    "‑": "-",
    "‒": "-",
    "–": "-",
    "—": "-",
    "−": "-"
}


def normalise_text_representation(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    for character in UNICODE_SPACE_CHARACTERS:
        text = text.replace(
            character,
            " "
        )

    for source, target in APOSTROPHE_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    for source, target in DASH_REPLACEMENTS.items():
        text = text.replace(
            source,
            target
        )

    text = text.replace(
        "\u00ad",
        ""
    )

    text = (
        text
        .replace("\r\n", "\n")
        .replace("\r", "\n")
    )

    normalised_lines = []

    for line in text.splitlines():

        line = re.sub(
            r"[ \t\f\v]+",
            " ",
            line
        ).rstrip()

        normalised_lines.append(
            line
        )

    text = "\n".join(
        normalised_lines
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return (
        text.strip()
        + "\n"
    )


In [6]:
# ============================================================
# 5. Apply Branch C normalisation to the COMPLETE frozen B representation
# ============================================================

NORMALISED_MARKDOWN = normalise_text_representation(
    SOURCE_B_MARKDOWN
)

if not NORMALISED_MARKDOWN.strip():
    raise ValueError(
        "D12 Branch C normalisation produced an empty representation."
    )

print(
    "Branch B characters:",
    len(SOURCE_B_MARKDOWN)
)

print(
    "Branch C characters:",
    len(NORMALISED_MARKDOWN)
)

print(
    "Representation changed:",
    SOURCE_B_MARKDOWN != NORMALISED_MARKDOWN
)


Branch B characters: 7754
Branch C characters: 7754
Representation changed: True


In [7]:
# ============================================================
# 6. Verify Branch C normalisation integrity
# ============================================================
#
# For this tabular document, integrity can be checked very strongly:
# every one of the 275 row identities, year values, emissions values
# and physical CSV-row provenance markers must remain identical.
# ============================================================

EXPECTED_NORMALISED_MARKDOWN = normalise_text_representation(
    SOURCE_B_MARKDOWN
)

deterministic_representation_verified = (
    NORMALISED_MARKDOWN
    == EXPECTED_NORMALISED_MARKDOWN
)


branch_c_rows = []

for line in NORMALISED_MARKDOWN.splitlines():

    match = TABLE_ROW_PATTERN.fullmatch(
        line.strip()
    )

    if match:
        branch_c_rows.append({
            "CSV data row":
                int(match.group(1)),

            "Year":
                int(match.group(2)),

            "Annual CO2 emissions":
                int(match.group(3))
        })


branch_c_row_count_valid = (
    len(branch_c_rows)
    == EXPECTED_SOURCE_ROW_COUNT
)

row_identity_and_order_preserved = (
    branch_c_rows
    == parent_rows
)

branch_c_years = [
    row["Year"]
    for row in branch_c_rows
]

branch_c_values = [
    row["Annual CO2 emissions"]
    for row in branch_c_rows
]

branch_c_source_rows = [
    row["CSV data row"]
    for row in branch_c_rows
]

year_sequence_preserved = (
    branch_c_years
    == parent_years
)

value_sequence_preserved = (
    branch_c_values
    == parent_values
)

source_row_sequence_preserved = (
    branch_c_source_rows
    == parent_source_rows
)

target_year_values_preserved = all(
    next(
        row["Annual CO2 emissions"]
        for row in branch_c_rows
        if row["Year"] == year
    )
    ==
    next(
        row["Annual CO2 emissions"]
        for row in parent_rows
        if row["Year"] == year
    )
    for year in TARGET_YEARS
)


normalisation_integrity_passed = bool(
    PARENT_EQUIVALENCE_PASSED
    and deterministic_representation_verified
    and branch_c_row_count_valid
    and row_identity_and_order_preserved
    and year_sequence_preserved
    and value_sequence_preserved
    and source_row_sequence_preserved
    and target_year_values_preserved
)


normalisation_check = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "parent_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "parent_row_count":
        len(parent_rows),

    "branch_C_row_count":
        len(branch_c_rows),

    "row_count_preserved":
        branch_c_row_count_valid,

    "row_identity_and_order_preserved":
        row_identity_and_order_preserved,

    "year_sequence_preserved":
        year_sequence_preserved,

    "value_sequence_preserved":
        value_sequence_preserved,

    "source_row_sequence_preserved":
        source_row_sequence_preserved,

    "target_year_values_preserved":
        target_year_values_preserved,

    "deterministic_representation_verified":
        deterministic_representation_verified,

    "complete_275_row_representation_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "target_year_filtering_applied":
        False,

    "row_reordering_applied":
        False,

    "unicode_nfkc_normalisation_applied":
        True,

    "unicode_space_standardisation_applied":
        True,

    "apostrophe_standardisation_applied":
        True,

    "dash_and_minus_standardisation_applied":
        True,

    "soft_hyphen_removal_applied":
        True,

    "line_endings_standardised":
        True,

    "horizontal_whitespace_normalisation_applied":
        True,

    "semantic_harmonisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_inference_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "numeric_rescaling_applied":
        False,

    "numeric_rounding_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_integrity_passed
}


NORMALISATION_CHECK_PATH.write_text(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        normalisation_check,
        ensure_ascii=False,
        indent=2
    )
)


if not normalisation_integrity_passed:
    raise ValueError(
        "D12 Branch C normalisation-integrity checks failed."
    )


{
  "document_id": "D12",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "parent_row_count": 275,
  "branch_C_row_count": 275,
  "row_count_preserved": true,
  "row_identity_and_order_preserved": true,
  "year_sequence_preserved": true,
  "value_sequence_preserved": true,
  "source_row_sequence_preserved": true,
  "target_year_values_preserved": true,
  "deterministic_representation_verified": true,
  "complete_275_row_representation_retained": true,
  "source_scope_filtering_applied": false,
  "target_year_filtering_applied": false,
  "row_reordering_applied": false,
  "unicode_nfkc_normalisation_applied": true,
  "unicode_space_standardisation_applied": true,
  "apostrophe_standardisation_applied": true,
  "dash_and_minus_standardisation_applied": true,
  "soft_hyphen_removal_applied": true,
  "line_endings_standardised": true,
  "horizontal_whitespace_normalisation_applied": true,
  "semantic_harmonisation_applied": false,
  "semantic_rewriting_appli

In [8]:
# ============================================================
# 7. Save Branch C normalised representation
# ============================================================

REPRESENTATION_PATH.write_text(
    NORMALISED_MARKDOWN,
    encoding="utf-8"
)

REPRESENTATION_SHA256 = sha256_file(
    REPRESENTATION_PATH
)

print(
    "Saved:",
    REPRESENTATION_PATH.name
)

print(
    "Representation SHA-256:",
    REPRESENTATION_SHA256
)


Saved: D12_branch_C_normalised_markdown.md
Representation SHA-256: d59ef1a0d81fa721474dde1caec0b92d48a489a191924043df164ba774aac4ad


In [9]:
# ============================================================
# 8. Create controlled Branch C extraction prompt
# ============================================================
#
# The substantive task and schema are frozen from Branch B.
# Only representation-dependent wording and branch identity are changed.
#
# Expected record/category counts are deliberately NOT disclosed.
# ============================================================

TARGET_YEAR_TEXT = "\n".join(
    f"- {year}"
    for year in TARGET_YEARS
)

BRANCH_C_PROMPT = f"""You are an information extraction assistant.

Extract the predefined annual CO2-emissions observations represented
in the attached deterministically normalised structural Markdown
representation of the original CSV.

Treat the attached deterministically normalised structural Markdown
representation as the only source of information.

The representation contains the complete annual time series with:

- CSV data row
- Year
- Annual CO2 emissions

The "CSV data row" column is structural provenance metadata identifying
the physical row of the original CSV, where the original header is
physical CSV row 1.

Extract one record for every year in the predefined target-year list
below.

Target years:

{TARGET_YEAR_TEXT}


For every included record return exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location


Use these fixed semantic-field values for every extracted record:

Category:
Environmental time-series

Topic:
Annual CO2 emissions

Description:
Annual CO2 emissions

Unit:
null


Field rules:

Category:
- Use exactly:
  "Environmental time-series"

Topic:
- Use exactly:
  "Annual CO2 emissions"

Description:
- Use exactly:
  "Annual CO2 emissions"

Value:
- Extract the value represented in the "Annual CO2 emissions"
  column for the corresponding target year.
- Preserve the source numerical value.
- Return the value as a JSON number.
- Do not calculate, interpolate, estimate, rescale, round or convert
  the source value.
- Do not use values from neighbouring years.

Unit:
- Return null.
- The supplied source does not explicitly represent a separate
  measurement-unit field.
- Do not infer or introduce a unit from external knowledge.

Reporting Period:
- Use the corresponding target year as a string.
- Example format:
  "1750"

Source Location:
- Use the explicit "CSV data row" value associated with the target year
  in the structural Markdown.
- Return exactly this format:
  "CSV data row N"
- Do not infer row numbers from the target-year-list position.


Extraction rules:

- Extract only the predefined target years.
- Return one record for every year in the target-year list.
- Do not omit a listed year.
- Do not return years outside the predefined list.
- Use only values explicitly represented in the structural Markdown.
- Preserve the direct year-to-value association.
- Do not calculate missing values.
- Do not interpolate between years.
- Do not aggregate years.
- Do not introduce measurement units that are absent from the source.
- Do not use external knowledge.
- Do not follow external links.
- Do not modify or normalise source numerical values.
- Verify that every listed target year has been processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{{
  "document_id": "D12",
  "branch": "C",
  "records": [
    {{
      "Category": "Environmental time-series",
      "Topic": "Annual CO2 emissions",
      "Description": "Annual CO2 emissions",
      "Value": null,
      "Unit": null,
      "Reporting Period": null,
      "Source Location": null
    }}
  ]
}}

Return only the JSON object.
"""


PROMPT_PATH.write_text(
    BRANCH_C_PROMPT,
    encoding="utf-8"
)

PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)

print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)


Prompt saved: D12_branch_C_prompt.txt
Prompt SHA-256: 2ad360189cf3749e016d72f30803a8853ab79fb4c6548fc4a4b03dced03cc854


In [10]:
# ============================================================
# 9. Create representation and pre-extraction metadata
# ============================================================

REPRESENTATION_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "parent_B_representation_file":
        BRANCH_B_REPRESENTATION_PATH.name,

    "parent_B_representation_sha256":
        UPLOADED_BRANCH_B_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "representation_type":
        (
            "Complete frozen Branch B structural Markdown table "
            "with deterministic non-semantic normalisation"
        ),

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "complete_275_row_representation_retained":
        True,

    "scope_enforced_by_prompt_not_representation_filtering":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "branch_B_regeneration_attempted":
        False,

    "normalisation_applied":
        True,

    "semantic_rewriting_applied":
        False,

    "unit_inference_applied":
        False,

    "unit_conversion_applied":
        False,

    "numeric_calculation_applied":
        False,

    "numeric_rescaling_applied":
        False,

    "numeric_rounding_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ]
}


REPRESENTATION_METADATA_PATH.write_text(
    json.dumps(
        REPRESENTATION_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_METADATA_PRE = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised structural Markdown table",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "direct_document_ingestion":
        False,

    "structural_conversion_applied":
        True,

    "structural_conversion_inherited_from_branch_B":
        True,

    "normalisation_applied":
        True,

    "complete_source_document_retained":
        True,

    "source_scope_filtering_applied":
        False,

    "target_year_filtering_applied":
        False,

    "reference_values_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "manual_response_repair_permitted":
        False,

    "expected_output_format":
        "JSON object",

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "execution_environment":
        "Independent ChatGPT conversation",

    "model":
        "GPT-5.5",

    "created_at":
        datetime.now().isoformat(),

    "python_version":
        sys.version,

    "platform":
        platform.platform(),

    "validation_status":
        "Pending independent Branch C extraction and Stage 4 Validation C"
}


EXPERIMENT_METADATA_PRE_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_METADATA_PRE,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D12",
  "document_name": "Our World in Data — Annual CO2 emissions time series",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D12 - ourworldindataCO2.csv",
  "source_sha256": "eb46e8e036c08032bbd18d234d3da22442111288e26af32d3eacef592457e788",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised structural Markdown table",
  "representation_file": "D12_branch_C_normalised_markdown.md",
  "representation_sha256": "d59ef1a0d81fa721474dde1caec0b92d48a489a191924043df164ba774aac4ad",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "direct_document_ingestion": false,
  "structural_conversion_applied": true,
  "structural_conversion_inherited_from_branch_B": true,
  "normalisation_applied": true,
  "complete_source_document_retained": true,
  "source_scope_filtering_applied": false,
  "target_year_filtering_applied": false,
  "reference_val

In [11]:
# ============================================================
# 10. Final pre-extraction control check
# ============================================================

PRECHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "source_identity_verified":
        SOURCE_HASH_MATCH,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "complete_275_row_representation_retained":
        True,

    "row_identity_and_order_preserved":
        row_identity_and_order_preserved,

    "year_sequence_preserved":
        year_sequence_preserved,

    "value_sequence_preserved":
        value_sequence_preserved,

    "source_row_sequence_preserved":
        source_row_sequence_preserved,

    "target_year_values_preserved":
        target_year_values_preserved,

    "representation_exists":
        REPRESENTATION_PATH.exists(),

    "prompt_exists":
        PROMPT_PATH.exists(),

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "reference_values_used_for_transformation":
        False,

    "ready_for_independent_llm_execution":
        bool(
            SOURCE_HASH_MATCH
            and PARENT_EQUIVALENCE_PASSED
            and normalisation_check[
                "normalisation_integrity_passed"
            ]
            and REPRESENTATION_PATH.exists()
            and PROMPT_PATH.exists()
        )
}


PRECHECK_PATH.write_text(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        PRECHECK,
        ensure_ascii=False,
        indent=2
    )
)


if not PRECHECK[
    "ready_for_independent_llm_execution"
]:
    raise ValueError(
        "D12 Branch C is not ready for independent LLM execution."
    )


{
  "document_id": "D12",
  "branch": "C",
  "source_identity_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "complete_275_row_representation_retained": true,
  "row_identity_and_order_preserved": true,
  "year_sequence_preserved": true,
  "value_sequence_preserved": true,
  "source_row_sequence_preserved": true,
  "target_year_values_preserved": true,
  "representation_exists": true,
  "prompt_exists": true,
  "expected_record_count_disclosed_to_model": false,
  "expected_category_counts_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [12]:
# ============================================================
# 11. Download pre-extraction Branch C artefacts
# ============================================================

for path in [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH
]:
    files.download(
        path
    )


print(
    "\nIndependent extraction instructions:\n"
    "1. Open a new independent ChatGPT conversation.\n"
    "2. Upload ONLY D12_branch_C_normalised_markdown.md.\n"
    "3. Submit D12_branch_C_prompt.txt exactly once.\n"
    "4. Do not upload the original CSV, Branch B artefacts, Stage 1 "
    "reference values, or previous extraction/validation outputs.\n"
    "5. Do not manually repair, correct, reorder, or regenerate the response.\n"
    "6. Save the complete first response exactly as returned in TXT."
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Independent extraction instructions:
1. Open a new independent ChatGPT conversation.
2. Upload ONLY D12_branch_C_normalised_markdown.md.
3. Submit D12_branch_C_prompt.txt exactly once.
4. Do not upload the original CSV, Branch B artefacts, Stage 1 reference values, or previous extraction/validation outputs.
5. Do not manually repair, correct, reorder, or regenerate the response.
6. Save the complete first response exactly as returned in TXT.


In [13]:
# ============================================================
# 12. Upload and preserve the complete raw Branch C response
# ============================================================

uploaded_response = files.upload()

if len(uploaded_response) != 1:
    raise ValueError(
        "Upload exactly one complete raw D12 Branch C response file."
    )


RAW_RESPONSE_SOURCE = Path(
    next(
        iter(
            uploaded_response
        )
    )
)


RAW_RESPONSE_TEXT = (
    RAW_RESPONSE_SOURCE.read_text(
        encoding="utf-8"
    )
)

if not RAW_RESPONSE_TEXT.strip():
    raise ValueError(
        "Uploaded D12 Branch C response is empty."
    )


RAW_RESPONSE_PATH.write_text(
    RAW_RESPONSE_TEXT,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)


print(
    "Raw response preserved unchanged."
)

print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)


Saving D12_branch_C_raw_response.txt to D12_branch_C_raw_response.txt
Raw response preserved unchanged.
Raw response SHA-256: a080bbd116a48c9ec2017ca6e743ef777d93f8f06f53fe2c7b9fcaa0d736a5c6


In [14]:
# ============================================================
# 13. Parse raw response WITHOUT repair
# ============================================================

valid_json = True
json_parsing_error = None
parsed_response = None


try:
    parsed_response = json.loads(
        RAW_RESPONSE_TEXT
    )

except json.JSONDecodeError as exc:
    valid_json = False
    json_parsing_error = str(exc)


top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)

document_id_correct = (
    top_level_object_valid
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)

branch_correct = (
    top_level_object_valid
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)

records_is_list = (
    top_level_object_valid
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)

records_evaluable = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
)

extracted_records = (
    parsed_response["records"]
    if records_evaluable
    else []
)

observed_record_count = (
    len(extracted_records)
    if records_evaluable
    else None
)


print("Valid JSON:", valid_json)
print("Records evaluable:", records_evaluable)
print("Observed records:", observed_record_count)

if json_parsing_error:
    print(
        "JSON parsing error:",
        json_parsing_error
    )


Valid JSON: True
Records evaluable: True
Observed records: 25


In [15]:
# ============================================================
# 14. Validate record schema and field types
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []


if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):
            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Record is not a JSON object"
            })

            continue


        observed_fields = list(
            record.keys()
        )

        if observed_fields != EXPECTED_FIELDS:
            record_structure_issues.append({
                "record_index":
                    record_index,

                "issue":
                    "Field names or field order differ",

                "expected_fields":
                    EXPECTED_FIELDS,

                "observed_fields":
                    observed_fields
            })


        for field in STRING_OR_NULL_FIELDS:

            value = record.get(field)

            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):
                field_type_issues.append({
                    "record_index":
                        record_index,

                    "field":
                        field,

                    "observed_type":
                        type(value).__name__
                })


        value = record.get("Value")

        if (
            isinstance(value, bool)
            or not isinstance(
                value,
                VALUE_ALLOWED_TYPES
            )
        ):
            field_type_issues.append({
                "record_index":
                    record_index,

                "field":
                    "Value",

                "observed_type":
                    type(value).__name__
            })


        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(field)

            if (
                value is None
                or (
                    isinstance(
                        value,
                        str
                    )
                    and not value.strip()
                )
            ):
                missing_mandatory_values.append({
                    "record_index":
                        record_index,

                    "field":
                        field
                })


record_schema_valid = (
    len(record_structure_issues) == 0
    if records_evaluable
    else None
)

field_types_valid = (
    len(field_type_issues) == 0
    if records_evaluable
    else None
)

mandatory_fields_complete = (
    len(missing_mandatory_values) == 0
    if records_evaluable
    else None
)


print("Record schema valid:", record_schema_valid)
print("Field types valid:", field_types_valid)
print("Mandatory fields complete:", mandatory_fields_complete)


Record schema valid: True
Field types valid: True
Mandatory fields complete: True


In [16]:
# ============================================================
# 15. Content/scope diagnostics kept separate from schema validity
# ============================================================

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get("Category")
            for record in extracted_records
            if isinstance(record, dict)
        )
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


    constant_fields_valid = all(
        (
            isinstance(record, dict)
            and record.get("Category")
                == REFERENCE_CATEGORY
            and record.get("Topic")
                == REFERENCE_TOPIC
            and record.get("Description")
                == REFERENCE_DESCRIPTION
            and record.get("Unit")
                is REFERENCE_UNIT
        )
        for record in extracted_records
    )


    observed_reporting_periods = [
        record.get("Reporting Period")
        for record in extracted_records
        if isinstance(record, dict)
    ]


    expected_reporting_periods = [
        str(year)
        for year in TARGET_YEARS
    ]


    reporting_period_counts = Counter(
        observed_reporting_periods
    )


    all_target_years_present_once = all(
        reporting_period_counts[
            str(year)
        ] == 1
        for year in TARGET_YEARS
    )


    no_unexpected_reporting_periods = (
        set(
            observed_reporting_periods
        )
        == set(
            expected_reporting_periods
        )
    )


    source_location_pattern = re.compile(
        r"^CSV data row (\d+)$"
    )


    source_location_format_valid = all(
        (
            isinstance(record, dict)
            and isinstance(
                record.get(
                    "Source Location"
                ),
                str
            )
            and source_location_pattern.fullmatch(
                record.get(
                    "Source Location"
                )
            )
            is not None
        )
        for record in extracted_records
    )


    year_source_row_mapping_valid = True

    for record in extracted_records:

        if not isinstance(record, dict):
            year_source_row_mapping_valid = False
            break

        try:
            year = int(
                record.get(
                    "Reporting Period"
                )
            )
        except (TypeError, ValueError):
            year_source_row_mapping_valid = False
            break

        expected_parent_row = next(
            (
                row["CSV data row"]
                for row in parent_rows
                if row["Year"] == year
            ),
            None
        )

        expected_location = (
            f"CSV data row {expected_parent_row}"
            if expected_parent_row is not None
            else None
        )

        if record.get(
            "Source Location"
        ) != expected_location:
            year_source_row_mapping_valid = False
            break


    duplicate_complete_record_signature_count = len([
        count
        for count in Counter(
            tuple(
                json.dumps(
                    record.get(field),
                    ensure_ascii=False,
                    sort_keys=True
                )
                for field in EXPECTED_FIELDS
            )
            for record in extracted_records
            if isinstance(record, dict)
        ).values()
        if count > 1
    ])


else:

    record_count_valid = None
    observed_category_counts = None
    category_counts_valid = None
    constant_fields_valid = None
    observed_reporting_periods = None
    all_target_years_present_once = None
    no_unexpected_reporting_periods = None
    source_location_format_valid = None
    year_source_row_mapping_valid = None
    duplicate_complete_record_signature_count = None


scope_complete = bool(
    record_count_valid
    and category_counts_valid
    and constant_fields_valid
    and all_target_years_present_once
    and no_unexpected_reporting_periods
    and source_location_format_valid
    and year_source_row_mapping_valid
) if records_evaluable else False


CONTENT_DIAGNOSTICS = {
    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches_reference":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match_reference":
        category_counts_valid,

    "constant_fields_valid":
        constant_fields_valid,

    "all_target_years_present_once":
        all_target_years_present_once,

    "no_unexpected_reporting_periods":
        no_unexpected_reporting_periods,

    "source_location_format_valid":
        source_location_format_valid,

    "year_source_row_mapping_valid":
        year_source_row_mapping_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "duplicate_complete_record_signature_count":
        duplicate_complete_record_signature_count
}


print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


{
  "expected_record_count": 25,
  "observed_record_count": 25,
  "record_count_matches_reference": true,
  "expected_category_counts": {
    "Environmental time-series": 25
  },
  "observed_category_counts": {
    "Environmental time-series": 25
  },
  "category_counts_match_reference": true,
  "constant_fields_valid": true,
  "all_target_years_present_once": true,
  "no_unexpected_reporting_periods": true,
  "source_location_format_valid": true,
  "year_source_row_mapping_valid": true,
  "mandatory_fields_complete": true,
  "duplicate_complete_record_signature_count": 0
}


In [17]:
# ============================================================
# 16. Determine technical/schema validity
# ============================================================
#
# IMPORTANT:
# Count/category/year/source-row diagnostics do NOT determine
# technical/schema validity.
# ============================================================

structure_valid = bool(
    valid_json
    and top_level_object_valid
    and document_id_correct
    and branch_correct
    and records_is_list
    and record_schema_valid is True
    and field_types_valid is True
)


STRUCTURE_CHECK = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        "Complete deterministically normalised structural Markdown table",

    "valid_json":
        bool(valid_json),

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        bool(top_level_object_valid),

    "document_id_correct":
        bool(document_id_correct),

    "branch_correct":
        bool(branch_correct),

    "records_is_list":
        bool(records_is_list),

    "records_evaluable":
        bool(records_evaluable),

    "record_schema_valid":
        record_schema_valid,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structure_valid":
        bool(structure_valid),

    "scope_complete":
        bool(scope_complete)
}


STRUCTURE_CHECK_PATH.write_text(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D12",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised structural Markdown table",
  "valid_json": true,
  "json_parsing_error": null,
  "top_level_object_valid": true,
  "document_id_correct": true,
  "branch_correct": true,
  "records_is_list": true,
  "records_evaluable": true,
  "record_schema_valid": true,
  "record_structure_issues": [],
  "field_types_valid": true,
  "field_type_issues": [],
  "content_diagnostics": {
    "expected_record_count": 25,
    "observed_record_count": 25,
    "record_count_matches_reference": true,
    "expected_category_counts": {
      "Environmental time-series": 25
    },
    "observed_category_counts": {
      "Environmental time-series": 25
    },
    "category_counts_match_reference": true,
    "constant_fields_valid": true,
    "all_target_years_present_once": true,
    "no_unexpected_reporting_periods": true,
    "source_location_format_valid"

In [18]:
# ============================================================
# 17. Preserve parsed extraction only when records are evaluable
# ============================================================

parsed_extraction_created = False
parsed_extraction_sha256 = None


if records_evaluable:

    canonical_extraction = {
        "document_id":
            DOCUMENT_ID,

        "branch":
            BRANCH,

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            canonical_extraction,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )

    parsed_extraction_created = True

    print(
        "Parsed extraction saved:",
        PARSED_EXTRACTION_PATH.name
    )

else:

    print(
        "No parsed extraction created because the preserved raw response "
        "does not contain an evaluable JSON records structure."
    )


Parsed extraction saved: D12_branch_C_parsed_extraction.json


In [19]:
# ============================================================
# 18. Create final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "structure_check_file":
        STRUCTURE_CHECK_PATH.name,

    "structure_valid":
        bool(structure_valid),

    "notes":
        (
            "Branch C applies deterministic non-semantic normalisation "
            "to the exact frozen Branch B complete 275-row structural "
            "Markdown. No rows or target years are filtered, and the "
            "year/value/source-row associations are preserved exactly. "
            "No unit inference, calculation, rescaling, rounding, "
            "semantic rewriting or reference-guided correction is applied. "
            "Accuracy is evaluated separately in Stage 4 Validation C."
        )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "parent_branch":
        PARENT_BRANCH,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        SOURCE_HASH_MATCH,

    "input_representation":
        "Complete deterministically normalised structural Markdown table",

    "representation_file":
        REPRESENTATION_PATH.name,

    "representation_sha256":
        REPRESENTATION_SHA256,

    "parent_B_equivalence_passed":
        PARENT_EQUIVALENCE_PASSED,

    "normalisation_integrity_passed":
        normalisation_check[
            "normalisation_integrity_passed"
        ],

    "structural_conversion_inherited_from_branch_B":
        True,

    "branch_B_regeneration_attempted":
        False,

    "normalisation_applied":
        True,

    "complete_source_document_retained":
        True,

    "target_year_filtering_applied":
        False,

    "reference_values_used_for_transformation":
        False,

    "expected_record_count_disclosed_to_model":
        False,

    "expected_category_counts_disclosed_to_model":
        False,

    "raw_response_preserved":
        True,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "valid_json":
        bool(valid_json),

    "records_evaluable":
        bool(records_evaluable),

    "record_schema_valid":
        record_schema_valid,

    "field_types_valid":
        field_types_valid,

    "structure_valid":
        bool(structure_valid),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "constant_fields_valid":
        constant_fields_valid,

    "all_target_years_present_once":
        all_target_years_present_once,

    "no_unexpected_reporting_periods":
        no_unexpected_reporting_periods,

    "source_location_format_valid":
        source_location_format_valid,

    "year_source_row_mapping_valid":
        year_source_row_mapping_valid,

    "scope_complete":
        bool(scope_complete),

    "duplicate_complete_record_signature_count":
        duplicate_complete_record_signature_count,

    "parsed_extraction_created":
        parsed_extraction_created,

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "accuracy_validation_completed":
        False,

    "validation_status":
        (
            "Pending Stage 4 Branch C validation against the fixed Stage 1 "
            "reference dataset using Branch A-frozen D12 comparison rules"
            if records_evaluable
            else
            "Not content-evaluable because the preserved Branch C response "
            "does not contain an evaluable JSON records structure"
        )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


{
  "document_id": "D12",
  "document_name": "Our World in Data — Annual CO2 emissions time series",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "parent_branch": "B",
  "source_file": "D12 - ourworldindataCO2.csv",
  "source_sha256": "eb46e8e036c08032bbd18d234d3da22442111288e26af32d3eacef592457e788",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised structural Markdown table",
  "representation_file": "D12_branch_C_normalised_markdown.md",
  "representation_sha256": "d59ef1a0d81fa721474dde1caec0b92d48a489a191924043df164ba774aac4ad",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "structural_conversion_inherited_from_branch_B": true,
  "branch_B_regeneration_attempted": false,
  "normalisation_applied": true,
  "complete_source_document_retained": true,
  "target_year_filtering_applied": false,
  "reference_values_used_for_transformation": false,
  "expected_record_count_disclosed_to_mod

In [20]:
# ============================================================
# 19. Final artefact inventory and downloads
# ============================================================

artefacts = [
    PARENT_CHECK_PATH,
    NORMALISATION_CHECK_PATH,
    REPRESENTATION_PATH,
    REPRESENTATION_METADATA_PATH,
    PROMPT_PATH,
    EXPERIMENT_METADATA_PRE_PATH,
    PRECHECK_PATH,
    RAW_RESPONSE_PATH,
    STRUCTURE_CHECK_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]

if parsed_extraction_created:
    artefacts.append(
        PARSED_EXTRACTION_PATH
    )


print(
    "Final D12 Branch C artefacts:"
)

for path in artefacts:
    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )


for path in artefacts:
    if path.exists():
        files.download(
            path
        )


Final D12 Branch C artefacts:
- D12_branch_C_parent_B_equivalence_check.json | exists: True
- D12_branch_C_normalisation_check.json | exists: True
- D12_branch_C_normalised_markdown.md | exists: True
- D12_branch_C_representation_metadata.json | exists: True
- D12_branch_C_prompt.txt | exists: True
- D12_branch_C_experiment_metadata_pre.json | exists: True
- D12_branch_C_pre_extraction_check.json | exists: True
- D12_branch_C_raw_response.txt | exists: True
- D12_branch_C_structure_check.json | exists: True
- D12_branch_C_experiment_metadata.json | exists: True
- D12_branch_C_experiment_summary.json | exists: True
- D12_branch_C_parsed_extraction.json | exists: True


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>